# Notebook 02 - Preprocessing

## Contexto

Este notebook aplica as decisoes tomadas no notebook 01 (EDA) para transformar
o dataset bruto em um conjunto adequado para modelagem no notebook 03.

## Ponto de partida

- Fonte: CSV bruto (`Base_M43_Pratique_Hypothyroid.csv`, 3772 linhas, 30 colunas)
- Optou-se por partir do dataset original, sem herdar pre-tratamentos do notebook 01,
para garantir disciplina de pipeline (evitar vazamentos de imputacao pre-split).

## Resumo das decisoes do EDA

### Variaveis Removidas

| Variavel | Motivo |
|----------|--------|
| TT4 | Multicolinearidade com FTI (corr 0.79), FTI tem maior poder preditivo (0.33 vs 0.29) |
| I131 treatment | Poder preditivo nulo, H2 refutada na camada 4 |
| query on thyroxine | Categoria rara, baixo poder preditivo |
| hypopituitary | Apenas 1 paciente |
| lithium | 18 pacientes, volume inviavel. Informacao capturada em psych_lithium_group |
| TBG / TBG measured | 100% nulos |
| pregnant | 100% positivo, leakage. Absorvida em risk_factors |
| goitre | 100% positivo, 34 pacientes. Absorvida em risk_factors |
| tumor | Proporcao identica ao target. Absorvida em risk_factors |

### Features de Engineering

| Feature | Composicao | Objetivo |
|---------|-----------|----------|
| risk_factors | pregnant + sick + thyroid surgery + goitre + tumor | Contagem 0-5 de condicoes de risco clinico |
| psych_lithium_group | psych + lithium | 0=sem psych, 1=psych sem litio, 2=psych com litio |
| num_exames_coletados | todas as colunas measured | Perfil de coleta laboratorial por paciente |

### Riscos Documentados

- Leakage potencial: on thyroxine, TSH measured, TT4 measured
- Desbalanceamento: 92.29% positivo, tratamento no notebook 03
- Unidades de medida nao confirmadas nos exames laboratoriais

## Pipeline de preprocessing

Este notebook encerra no estado pos-deterministico e pre-estatistico. As operacoes
que aprendem parametros de conjunto (OneHotEncoder, RobustScaler, KNNImputer) foram
movidas para o Pipeline do notebook 03, onde rodam por fold dentro do cross-validation
para evitar vazamento entre treino e validacao.

Sequencia de operacoes deste notebook:

1. Limpeza de `?` para NaN + conversao numerica das colunas de exames
2. Encoding do target
3. Encoding das binarias
4. Feature engineering
5. Drops
6. Transformacao log1p
7. Tratamento de outlier de age (regra deterministica, limiar fixo)
8. Split treino/teste (estratificado)
9. Export do artefato intermediario

Operacoes delegadas ao Pipeline do notebook 03 (rodam por fold no CV):

- OneHotEncoder no referral_source
- RobustScaler nas variaveis continuas
- KNNImputer para valores faltantes

*Nota: a remocao de features de leakage (num_exames_coletados, TSH measured,
TT4 measured) e o uso de KNNImputer surgiram durante a analise da etapa 4.
Ver secao "Reformulacao do pipeline" para o historico completo.*

## Referencia

- Notebook 01 (01_eda.ipynb): analise exploratoria completa e decisoes iniciais

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split


In [2]:
# carregar base raw
df = pd.read_csv('Base_M43_Pratique_Hypothyroid.csv')
# visualiza DataFrame
display(df.head(), df.tail())

,age,sex,on thyroxine,query on thyroxine,on antithyroid medication,sick,pregnant,thyroid surgery,I131 treatment,query hypothyroid,...,TT4 measured,TT4,T4U measured,T4U,FTI measured,FTI,TBG measured,TBG,referral source,binaryClass
0,41,F,f,f,f,f,f,f,f,f,...,t,125,t,1.14,t,109,f,?,SVHC,P
1,23,F,f,f,f,f,f,f,f,f,...,t,102,f,?,f,?,f,?,other,P
2,46,M,f,f,f,f,f,f,f,f,...,t,109,t,0.91,t,120,f,?,other,P
3,70,F,t,f,f,f,f,f,f,f,...,t,175,f,?,f,?,f,?,other,P
4,70,F,f,f,f,f,f,f,f,f,...,t,61,t,0.87,t,70,f,?,SVI,P


,age,sex,on thyroxine,query on thyroxine,on antithyroid medication,sick,pregnant,thyroid surgery,I131 treatment,query hypothyroid,...,TT4 measured,TT4,T4U measured,T4U,FTI measured,FTI,TBG measured,TBG,referral source,binaryClass
3767,30,F,f,f,f,f,f,f,f,f,...,f,?,f,?,f,?,f,?,other,P
3768,68,F,f,f,f,f,f,f,f,f,...,t,124,t,1.08,t,114,f,?,SVI,P
3769,74,F,f,f,f,f,f,f,f,f,...,t,112,t,1.07,t,105,f,?,other,P
3770,72,M,f,f,f,f,f,f,f,f,...,t,82,t,0.94,t,87,f,?,SVI,P
3771,64,F,f,f,f,f,f,f,f,f,...,t,99,t,1.07,t,92,f,?,other,P


In [3]:
# validação do shape do dataframe
df.shape

(3772, 30)

## Etapa 1 - Limpeza de Representacao de Nulos

O dataset original utiliza o caractere `?` para representar valores ausentes em vez do padrao NaN.
Esta etapa corrige essa representacao e converte as colunas de exames laboratoriais para dtype numerico.

**Colunas convertidas:** age, TSH, T3, TT4, T4U, FTI

**Coluna excluida da conversao:** TBG (100% nula, sera removida na etapa de drops)

**Justificativa:** a conversao numerica e pre-requisito para operacoes posteriores
(log1p, scaling, imputacao). Sem ela, as colunas permanecem como dtype object
e operacoes matematicas falham.

**Validacao esperada:** colunas de exames com dtype float64, valores `?` substituidos por NaN.

In [4]:
# replace das caracteres ? como nulos no df tratado anteriomente para remoção das
df = df.replace('?', np.nan)
#transforma em numericos
cols_numericas = ['age', 'TSH', 'T3', 'TT4', 'T4U', 'FTI']
df[cols_numericas] = df[cols_numericas].apply(pd.to_numeric, errors='coerce')

/tmp/ipykernel_2232/3666465304.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace('?', np.nan)


In [5]:
# valida limpeza
df.isna().sum()

,0
age,1
sex,150
on thyroxine,0
query on thyroxine,0
on antithyroid medication,0
sick,0
pregnant,0
thyroid surgery,0
I131 treatment,0
query hypothyroid,0


## Etapa 2 - Encoding do Target (binaryClass)

Transformacao dos valores categoricos P (positivo) e N (negativo) para representacao numerica binaria.

| Valor Original | Valor Numerico | Significado |
|---------------|----------------|-------------|
| P | 1 | Hipertireoidismo presente |
| N | 0 | Ausencia de hipertireoidismo |

**Justificativa:** algoritmos de classificacao exigem target numerico. A convencao padrao atribui 1 a classe de interesse (doenca presente) e 0 a ausencia.

**Validacao esperada:** contagens devem se manter identicas as originais (P=3481, N=291).

In [6]:
# contagem de unidade para o encoding e dtype de binaryClass antes do enconding
display(df['binaryClass'].value_counts())
display(df['binaryClass'].dtype)

,count
binaryClass,
P,3481
N,291


dtype('O')

In [7]:
# enconding da variavel target
df['binaryClass'] = df['binaryClass'].map({'N': 0, 'P': 1})

In [8]:
# contagem de unidade para o encoding e dtype de binaryClass depois do enconding
display(df['binaryClass'].value_counts())
display(df['binaryClass'].dtype)

,count
binaryClass,
1,3481
0,291


dtype('int64')

## Etapa 3 - Encoding das Variaveis Binarias

Transformacao das variaveis categoricas binarias de string para representacao numerica
por mapeamento fixo (deterministico, sem dependencia estatistica).

**Mapeamento t/f (1/0):**

on thyroxine, query on thyroxine, on antithyroid medication, sick, pregnant,
thyroid surgery, I131 treatment, query hypothyroid, query hyperthyroid, psych,
lithium, goitre, tumor, TSH measured, TT4 measured, T3 measured,
T4U measured, FTI measured, TBG measured

**Mapeamento M/F (0/1):**

sex (atribuicao convencional, sem relacao ordinal)

**Justificativa:** mapeamento aplicado antes do split por ser transformacao deterministica
(regra fixa, sem calculo estatistico). Variaveis que serao dropadas posteriormente
tambem sao convertidas, pois o feature engineering (etapa seguinte) depende
de algumas delas em formato numerico.

**Observacao:** referral_source nao e afetada por este mapeamento.
Seu encoding (OneHotEncoder) ocorre apos o split (etapa 8).

In [9]:
# dicionario para replace t/f e M/F
map_encoding = {'f':0, 't':1, 'M':0, 'F':1}

# replace dos string
df = df.replace(map_encoding).infer_objects(copy=False)

#visualiza dataframe numerico
display(df.head(), df.tail())

/tmp/ipykernel_2232/2166583910.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(map_encoding).infer_objects(copy=False)


,age,sex,on thyroxine,query on thyroxine,on antithyroid medication,sick,pregnant,thyroid surgery,I131 treatment,query hypothyroid,...,TT4 measured,TT4,T4U measured,T4U,FTI measured,FTI,TBG measured,TBG,referral source,binaryClass
0,41.0,1.0,0,0,0,0,0,0,0,0,...,1,125.0,1,1.14,1,109.0,0,NaN,SVHC,1
1,23.0,1.0,0,0,0,0,0,0,0,0,...,1,102.0,0,NaN,0,NaN,0,NaN,other,1
2,46.0,0.0,0,0,0,0,0,0,0,0,...,1,109.0,1,0.91,1,120.0,0,NaN,other,1
3,70.0,1.0,1,0,0,0,0,0,0,0,...,1,175.0,0,NaN,0,NaN,0,NaN,other,1
4,70.0,1.0,0,0,0,0,0,0,0,0,...,1,61.0,1,0.87,1,70.0,0,NaN,SVI,1


,age,sex,on thyroxine,query on thyroxine,on antithyroid medication,sick,pregnant,thyroid surgery,I131 treatment,query hypothyroid,...,TT4 measured,TT4,T4U measured,T4U,FTI measured,FTI,TBG measured,TBG,referral source,binaryClass
3767,30.0,1.0,0,0,0,0,0,0,0,0,...,0,NaN,0,NaN,0,NaN,0,NaN,other,1
3768,68.0,1.0,0,0,0,0,0,0,0,0,...,1,124.0,1,1.08,1,114.0,0,NaN,SVI,1
3769,74.0,1.0,0,0,0,0,0,0,0,0,...,1,112.0,1,1.07,1,105.0,0,NaN,other,1
3770,72.0,0.0,0,0,0,0,0,0,0,0,...,1,82.0,1,0.94,1,87.0,0,NaN,SVI,1
3771,64.0,1.0,0,0,0,0,0,0,0,0,...,1,99.0,1,1.07,1,92.0,0,NaN,other,1


## Etapa 4 - Feature Engineering

Construção de features intencionadas à agregar condições clínicas raras e com leakage, captura gradação do efeito psiquiátrico no TSH e
captura perfil de coleta laboratorial.

| Feature | Variáveis | Objetivo |
|---------|-----------|----------|
| `risk_factors` | pregnant, sick, thyroid surgery, goitre, tumor | Perfil de risco clínico agregado |
| `psych_lithium_group` | psych, lithium | Gradação do efeito psiquiátrico no TSH |
| `num_exames_coletados` | todas as measured | Perfil de coleta laboratorial |

**Obs:** versão iniciada, reformulada a seguir

**Justificativa:** A identificação de riscos para o modelo, que podem diminuir assertividade, gerar leakage e multicolineariedade, e ainda assim manter fatores colaborativos entre as features que somam poder preditivo ao modelo final

**Validacao esperada:** geração de novas features no DataFrame ( shape: 3772 , 33) `risk_factors` valores entre (0-5). `psych_lithium_group` valores entre (0-2). `num_exames_coletados` valores entre (0-5)




In [10]:
# Criação de risk_factors realizando a soma linear das features pregnant, sick, thyroid surgery, goitre e tumor
select_features = df[['pregnant', 'sick', 'thyroid surgery', 'goitre', 'tumor']] # definição das colunas

#Conta os sintomas por linha (axis=1) e cria uma nova coluna [risk_factors]
df['risk_factors'] = select_features.sum(axis=1, skipna=True)

In [11]:
# groupby para visualizar a proporção dos grupos em relação aos numeros dos exames
group_risk = df.groupby(['risk_factors','binaryClass',])[['TSH','T3','TT4','FTI']].mean()
# visualiza o grupo
display(group_risk)
print(df['risk_factors'].value_counts())

TSH        T3         TT4         FTI
risk_factors binaryClass                                             
0            0            40.923019  1.472566   72.470943   72.442339
             1             1.832238  2.044768  110.861130  114.255607
1            0            17.519048  1.500000   78.523810   84.333333
             1             2.540488  2.185306  116.163880  108.574830
2            1             1.029333  2.711765  125.666667  100.666667

risk_factors
0    3409
1     343
2      20
Name: count, dtype: int64


In [12]:
# visualiza distribuição da variavel risk_factors
pd.crosstab(df['risk_factors'], df['binaryClass'], normalize='index')

binaryClass,0,1
risk_factors,,
0,0.079202,0.920798
1,0.061224,0.938776
2,0.000000,1.000000


In [13]:
# Criação de risk_factors realizando a soma linear das features pregnant, sick, thyroid surgery, goitre e tumor
select_clinicalBurden = df[['pregnant', 'sick', 'thyroid surgery', 'goitre', 'tumor','on antithyroid medication', 'query hyperthyroid']] # definição das colunas

#Conta os sintomas por linha (axis=1) e atualiza uma nova coluna [risk_factors]
df['risk_factors'] = select_clinicalBurden.sum(axis=1, skipna=True)

In [14]:
# groupby para visualizar a proporção dos grupos em relação aos numeros dos exames
group_clinical = df.groupby(['risk_factors','binaryClass',])[['TSH','T3','TT4','FTI']].mean()
# visualiza o grupo
display(group_clinical)
print(df['risk_factors'].value_counts())

TSH        T3         TT4         FTI
risk_factors binaryClass                                             
0            0            41.239667  1.452830   71.943200   72.170386
             1             1.830332  2.007086  109.888041  113.424116
1            0            24.600000  1.596875   79.800000   81.062500
             1             2.367344  2.263255  117.620767  114.923611
2            0            40.000000  2.200000   75.000000   74.000000
             1             1.437358  2.873585  133.140351  113.964286
3            1             0.208750  3.550000  177.000000  114.000000

risk_factors
0    3182
1     521
2      65
3       4
Name: count, dtype: int64


## Reformulacao da risk_factors

### Analise da versao inicial

A primeira versao da feature (risk_factors) somava pregnant, sick, thyroid surgery, goitre e tumor.
Objetivo declarado: agregar condicoes clinicas raras, reduzir leakage de variaveis 100% positivo
e reduzir dimensionalidade.

Resultado da analise:
- Distribuicao concentrada em 0 (90.4% dos pacientes)
- risk_factors=1: 343 pacientes, 93.8% positivo (proximo ao baseline de 92.29%)
- risk_factors=2: 20 pacientes, 100% positivo

Diagnostico: o leakage nao foi eliminado, apenas transferido para o nivel agregado.
A composicao mantinha variaveis individualmente 100% positivo (pregnant, goitre),
o que reproduziu o problema num subgrupo.

### Redefinicao do objetivo

A feature original tentava simultaneamente medir "gravidade" e "risco".
Gravidade e captura pelas variaveis continuas dos exames (TSH, T3, TT4, FTI).
A feature binaria agregada faz mais sentido como **indicador de burden clinico**:
contagem de condicoes co-existentes documentadas.

### Nova composicao: clinical_burden

Composicao ampliada com 7 variaveis binarias:

**Incluidas:** sick, thyroid surgery, pregnant, tumor, goitre, on antithyroid medication, query hyperthyroid

**Excluidas com justificativa:**

| Variavel | Motivo |
|----------|--------|
| query hypothyroid | Direcao clinica contraria (suspeita de hipo, nao hiper) |
| psych | Ja capturada em psych_lithium_group |
| lithium, hypopituitary, I131 treatment | Ja descartadas em analise anterior |

### Hipotese a validar

Com composicao mais ampla, buckets de contagem devem se distribuir por mais niveis (0-7).
Pacientes com pregnant=1 ou goitre=1 (individualmente 100% positivo) serao misturados
com pacientes de outras condicoes no mesmo bucket, diluindo o efeito de leakage.

### Criterio de aceitacao

- Se algum bucket com mais de 5 pacientes continuar 100% positivo, diluicao falhou
- Se todos os buckets tem mistura de classes, diluicao funcionou

### Analise pos-validacao

Distribuicao dos buckets:

| Bucket | Pacientes | % do dataset |
|--------|-----------|--------------|
| 0 | 3182 | 84.4% |
| 1 | 521 | 13.8% |
| 2 | 65 | 1.7% |
| 3 | 4 | 0.1% |

Discriminacao por classe (TSH medio):

| Bucket | TSH neg | TSH pos | Observacao |
|--------|---------|---------|------------|
| 0 | 41.24 | 1.83 | Grande separacao, 84% do dataset |
| 1 | 24.60 | 2.37 | Separacao mantida |
| 2 | 40.00 | 1.44 | **Ambas as classes presentes (diluicao funcionou)** |
| 3 | - | 0.21 | 100% positivo, apenas 4 pacientes |

**Resultado:** hipotese validada. O bucket anteriormente problematico (contagem=2,
100% positivo com 20 pacientes na versao inicial) agora contem ambas as classes com
65 pacientes. O unico bucket 100% positivo remanescente (contagem=3) tem apenas
4 pacientes, abaixo do criterio de aceitacao (>5 pacientes).

**Observacao clinica:** o bucket 3 apresenta o menor TSH medio observado (0.21),
coerente com hipertireoidismo mais grave em pacientes com mais condicoes co-existentes.
Amostra pequena impede generalizacao.

**Decisao:** feature aceita para o pipeline. Nome mantido como `risk_factors`
por consistencia com iteracoes anteriores do notebook.

### Feature 2 - psych_lithium_group

Feature categorica que captura a gradacao do efeito psiquiatrico/psicofarmacologico sobre o TSH.

**Regras de construcao:**

| Valor | Condicao | Significado |
|-------|----------|-------------|
| 0 | psych=0 | Paciente sem condicao psiquiatrica documentada |
| 1 | psych=1 e lithium=0 | Paciente com condicao psiquiatrica, sem uso de litio |
| 2 | psych=1 e lithium=1 | Paciente com condicao psiquiatrica em uso de litio |

**Justificativa:** a analise multivariada do EDA identificou que psych e lithium apresentam
efeitos independentes de reducao do TSH. Como o TSH e o principal preditor do modelo
(correlacao -0.43 com o target), preservar essa gradacao de efeito e clinicamente relevante.
Agrupar as duas variaveis numa unica feature categorica evita multicolinearidade e captura
o efeito acumulado (nenhum, um, ambos).

**Observacao sobre lithium:** a variavel lithium foi descartada individualmente (apenas
18 pacientes, inviavel para modelagem). Sua informacao e preservada exclusivamente
atraves desta feature.

**Validacao esperada:** valores no intervalo [0, 2], distribuicao concentrada em 0
(a maioria dos pacientes nao apresenta condicao psiquiatrica), verificar se lithium=1
implica psych=1 (senao a categoria 2 nao existe corretamente).

In [15]:
# Define a lista de condições (use parênteses para cada comparação)
condicoes_psych = [
    (df['psych'] == 0) & (df['lithium'] == 0),  # Se 0 e 0
    (df['psych'] == 1) & (df['lithium'] == 1),   # Se 1 e 1
    (df['psych'] == 1) | (df['lithium'] == 1)  # Se ao menos um deles = 1
]

# Define a lista de resultados na mesma ordem das condições
escolhas_psych = [0, 2, 1]

df['psych_lithium_group'] = np.select(condicoes_psych, escolhas_psych, default=-1)


print(df['psych_lithium_group'].unique())
display(df['psych_lithium_group'].value_counts())


[0 1 2]


,count
psych_lithium_group,
0,3573
1,196
2,3


In [16]:
# visualiza distribuição da variavel psych_lithium_group
pd.crosstab(df['psych_lithium_group'], df['binaryClass'], normalize='index')

binaryClass,0,1
psych_lithium_group,,
0,0.078925,0.921075
1,0.045918,0.954082
2,0.000000,1.000000


### Analise pos-validacao - psych_lithium_group

Distribuicao das categorias:

| Categoria | Pacientes | % do dataset |
|-----------|-----------|--------------|
| 0 (nenhuma evidencia) | 3573 | 94.7% |
| 1 (evidencia parcial) | 196 | 5.2% |
| 2 (evidencia completa) | 3 | 0.08% |

Discriminacao por classe (crosstab normalizado):

| Categoria | % negativo | % positivo | Baseline (92.29%) |
|-----------|------------|------------|-------------------|
| 0 | 7.9% | 92.1% | coincide com baseline |
| 1 | 4.6% | 95.4% | +3.15 pontos acima |
| 2 | 0.0% | 100.0% | 100% positivo com 3 pacientes |

**Achado clinico:** dos 18 pacientes com lithium=1, apenas 3 tem psych=1 registrado.
Os outros 15 caem na categoria 1 (lithium sem psych documentado), validando a decisao
de usar a logica para todas as categorias intermediarias.

**Criterio de aceitacao:** categoria 2 tem 3 pacientes, abaixo do limite de 5 estabelecido
para caracterizar leakage. Passa no criterio.

**Poder discriminativo:** limitado. A categoria 0 (94.7% dos pacientes) coincide com o
baseline, entao a feature nao contribui para discriminar a maioria dos casos. Categoria 1
apresenta leve elevacao (+3 pontos) sobre o baseline, cobrindo 5% do dataset.

**Decisao:** feature mantida para teste no notebook 03. Importancia relativa sera
avaliada durante o treinamento.

### Feature 3 - num_exames_coletados

Contagem de exames laboratoriais coletados para cada paciente, capturando o perfil
de investigacao clinica realizada.

**Composicao:** soma das variaveis measured das 5 colunas de exames laboratoriais.

**Variaveis incluidas:** TSH measured, T3 measured, TT4 measured, T4U measured, FTI measured

**Variavel excluida:** TBG measured (constante em 0 no dataset, sera dropada)

**Justificativa:** o EDA identificou que pacientes com menos exames coletados (particularmente
os 369 pacientes sem TSH medido) apresentam padrao clinico distinto (100% positivo,
maioria referral_source=other). A quantidade de exames coletados reflete a intensidade
da investigacao clinica e pode capturar sinal indireto sobre o encaminhamento e a
suspeita inicial do medico solicitante.

**Diferenca em relacao as measured individuais:** as colunas measured serao mantidas
individualmente no dataset (nao ha drop). Esta feature adiciona a informacao agregada
sem eliminar a granularidade original. Cabe ao modelo (notebook 03) decidir se o valor
agregado ou as individuais tem maior contribuicao.

**Validacao esperada:** valores no intervalo [0, 5], sem NaN, distribuicao concentrada
no valor maximo (a maioria dos pacientes tem todos os exames coletados) com
subgrupos menores em valores baixos.

In [17]:
# Criação de num_exames_coletados realizando a soma das features TSH measured, T3 measured, TT4 measured, T4U measured, FTI measured
select_exams = df[['TSH measured', 'T3 measured', 'TT4 measured', 'T4U measured', 'FTI measured']] # definição das colunas

#Conta os sintomas por linha (axis=1) e cria uma nova coluna [num_exames_coletados]
df['num_exames_coletados'] = select_exams.sum(axis=1, skipna=True)

In [18]:
# groupby para visualizar a proporção dos grupos em relação aos numeros dos exames
group_exams = df.groupby(['num_exames_coletados','binaryClass',])[['TSH','T3','TT4','FTI']].mean()
# visualiza o grupo
display(group_exams)
print(df['num_exames_coletados'].value_counts())

TSH        T3         TT4         FTI
num_exames_coletados binaryClass                                             
0                    1                  NaN       NaN         NaN         NaN
1                    0            24.566667       NaN         NaN         NaN
                     1             2.145625  2.079167   98.000000         NaN
2                    0            20.000000  2.250000         NaN         NaN
                     1             1.936000  2.340000  119.000000         NaN
3                    0            29.380750  1.500000   68.365000         NaN
                     1             2.405649  2.047692  101.666667  107.173469
4                    0            28.927907       NaN   89.790698   87.418605
                     1             2.071856  2.464615  116.267748  117.769231
5                    0            42.474888  1.465471   70.069507   70.514350
                     1             1.835125  2.050431  111.340055  113.086595

num_exames_coletados
5    2752
4     537
3     247
0     179
1      44
2      13
Name: count, dtype: int64


In [19]:
# visualiza distribuição da variavel psych_lithium_group
pd.crosstab(df['num_exames_coletados'], df['binaryClass'], normalize='index')

binaryClass,0,1
num_exames_coletados,,
0,0.000000,1.000000
1,0.068182,0.931818
2,0.153846,0.846154
3,0.080972,0.919028
4,0.080074,0.919926
5,0.081032,0.918968


### Investigacao: perfil dos pacientes com num_exames_coletados=0

O grupo de 179 pacientes sem nenhum exame laboratorial coletado apresenta 100% de
positividade (binaryClass=1). Antes de decidir sobre o destino da feature
num_exames_coletados e das colunas measured individuais, e necessario entender
se esse padrao reflete:

- Artefato de amostragem (leakage a ser eliminado)
- Padrao clinico legitimo (diagnostico feito sem exames por algum protocolo)

**Metodo:** comparacao multivariada entre dois subgrupos do dataset:

- Grupo A: pacientes com num_exames_coletados=0 (n=179)
- Grupo B: demais pacientes (n=3593)

**Variaveis analisadas:**

| Variavel | Tipo | Metrica |
|----------|------|---------|
| referral_source | Categorica | Proporcao de cada categoria |
| on antithyroid medication | Binaria | Proporcao de tratados |
| age | Continua | Media |
| risk_factors | Ordinal | Media |

**Hipoteses a testar:**

1. Os 179 pacientes vem majoritariamente de referral_source=other (confirmar achado do EDA)
2. Ha maior proporcao de pacientes ja em tratamento com antithyroid medication no grupo A (diagnostico previo)
3. O perfil de risk_factors do grupo A e distinto do resto do dataset
4. Existem diferencas demograficas (age, sex) entre os grupos

**Criterio de decisao:**

- Se ha diferencas claras que justifiquem clinicamente o padrao, a feature pode ser mantida com documentacao
- Se nao ha caracteristica clinica distintiva, o padrao e artefato de amostragem e a feature deve ser removida

### Resultado da investigacao

**Comparacao das variaveis binarias e continuas:**

| Variavel | Grupo A (sem exames) | Grupo B (com exames) | Diferenca |
|----------|---------------------|---------------------|-----------|
| on antithyroid medication | 2.23% | 1.09% | +2x taxa |
| age (media) | 44.28 anos | 52.10 anos | -8 anos |
| risk_factors (media) | 0.25 | 0.17 | +47% |

**Distribuicao de referral_source:**

| Categoria | Grupo A | Grupo B |
|-----------|---------|---------|
| other | 99.44% | 56.30% |
| SVHC | 0.56% | 10.72% |
| SVI | 0.00% | 28.78% |
| STMW | 0.00% | 3.12% |
| SVHD | 0.00% | 1.09% |

**Avaliacao das hipoteses:**

- **H1 confirmada.** 99.44% dos pacientes do grupo A vem de referral_source=other,
contra 56.30% do grupo B. Diferenca massiva. Todas as demais fontes conhecidas
(SVI, STMW, SVHD) tem representacao zero no grupo A.
- **H2 parcialmente confirmada.** Grupo A tem o dobro da taxa de tratamento com
antithyroid medication, mas em valores absolutos ainda e baixa (2.23%). Nao
explica sozinha o padrao 100% positivo.
- **H3 confirmada.** Grupo A tem risk_factors 47% maior. Sinaliza mais burden
clinico documentado.
- **H4 confirmada.** Grupo A e ~8 anos mais jovem em media. Diferenca substancial
e clinicamente atipica (hipertireoidismo tem maior incidencia em mulheres de
meia-idade).

In [20]:
# separação dos grupos
grupo_A = df[df['num_exames_coletados'] == 0] #grupo com 0 exames
grupo_B = df[df['num_exames_coletados'] != 0] #grupos com pelo menos 1 coletado

#criação do dataframe
df_investigacao = pd.DataFrame({
    'grupo_A':[grupo_A['on antithyroid medication'].mean(),
               grupo_A['age'].mean(),
               grupo_A['risk_factors'].mean()],

    'grupo_B': [grupo_B['on antithyroid medication'].mean(),
                grupo_B['age'].mean(),
                grupo_B['risk_factors'].mean()]
}, index=['on antithyroid medication', 'age','risk_factors']
)
display(df_investigacao)

,grupo_A,grupo_B
on antithyroid medication,0.022346,0.010854
age,44.284916,52.107183
risk_factors,0.251397,0.172001


In [21]:
# visualiza referal source com base nos grupos
grupo_A['referral source'].value_counts(normalize=True)


,proportion
referral source,
other,0.994413
SVHC,0.005587


In [22]:
# visualiza referal source com base nos grupos
grupo_B['referral source'].value_counts(normalize=True)

,proportion
referral source,
other,0.563039
SVI,0.287782
SVHC,0.107153
STMW,0.031172
SVHD,0.010854


In [23]:
# verifica proporção por classe de tsh measured e tt4 measured para veirificar potencial de leakege
display(pd.crosstab(df['TSH measured'], df['binaryClass'], normalize='index'))
display(pd.crosstab(df['TT4 measured'], df['binaryClass'], normalize='index'))
#contabiliza, numeros por classes
display(pd.crosstab(df['TSH measured'], df['binaryClass']))
display(pd.crosstab(df['TT4 measured'], df['binaryClass']))

binaryClass,0,1
TSH measured,,
0,0.000000,1.000000
1,0.085513,0.914487


binaryClass,0,1
TT4 measured,,
0,0.021645,0.978355
1,0.080768,0.919232


binaryClass,0,1
TSH measured,,
0,0,369
1,291,3112


binaryClass,0,1
TT4 measured,,
0,5,226
1,286,3255


### Analise pos-validacao - num_exames_coletados

**Distribuicao dos buckets:**

| Bucket | Pacientes | % do dataset |
|--------|-----------|--------------|
| 0 | 179 | 4.7% |
| 1 | 44 | 1.2% |
| 2 | 13 | 0.3% |
| 3 | 247 | 6.5% |
| 4 | 537 | 14.2% |
| 5 | 2752 | 73.0% |

**Discriminacao por classe (proporcao de positivos):**

| Bucket | % negativo | % positivo | Comparacao com baseline (92.29%) |
|--------|-----------|-----------|----------------------------------|
| 0 | 0.00% | 100.00% | +7.7 pontos, 179 pacientes |
| 1 | 6.82% | 93.18% | proximo ao baseline |
| 2 | 15.38% | 84.62% | -7.7 pontos, 13 pacientes |
| 3 | 8.10% | 91.90% | proximo ao baseline |
| 4 | 8.01% | 91.99% | proximo ao baseline |
| 5 | 8.10% | 91.90% | proximo ao baseline |

**Achado critico:** o bucket 0 (179 pacientes sem nenhum exame coletado) apresenta
100% de positividade. Isso viola gravemente o criterio de aceitacao estabelecido
(bucket com mais de 5 pacientes 100% positivo indica leakage).

### Investigacao dos 179 pacientes

Comparacao multivariada entre o grupo sem exames (n=179) e o resto do dataset (n=3593):

| Variavel | Grupo sem exames | Resto do dataset | Diferenca |
|----------|------------------|------------------|-----------|
| on antithyroid medication | 2.23% | 1.09% | +2x taxa |
| age (media) | 44.28 | 52.10 | -8 anos |
| risk_factors (media) | 0.25 | 0.17 | +47% |

**Distribuicao de referral_source:**

| Categoria | Grupo sem exames | Resto do dataset |
|-----------|------------------|------------------|
| other | 99.44% | 56.30% |
| SVHC | 0.56% | 10.72% |
| SVI | 0.00% | 28.78% |
| STMW | 0.00% | 3.12% |
| SVHD | 0.00% | 1.09% |

### Interpretacao

O padrao "sem exames = 100% positivo" esta fortemente associado a fonte de
encaminhamento (referral source=other em 99.44% dos casos). Isso caracteriza
**artefato de coleta de dados**, nao decisao clinica documentavel. O dataset
provavelmente agregou registros de fontes com protocolos diferentes, e uma
subcategoria dentro de "other" registra pacientes sem exames laboratoriais
com diagnostico previo.

Manter esta feature no modelo criaria atalho para o algoritmo aprender uma regra
que nao generaliza para producao real ("sem exames + other = positivo").

### Decisao

- Feature `num_exames_coletados` **descartada**.
- Drop consolidado na etapa 5.
- Colunas measured individuais (TSH measured, TT4 measured) tambem apresentam
leakage confirmado (369 e 231 pacientes 100% e 97.8% positivos respectivamente)
e serao descartadas na etapa 5.
- Colunas measured sem leakage individual (T3 measured, T4U measured, FTI measured)
serao mantidas para avaliacao no notebook 03.

### Encaminhamento para notebook 04

Analise estratificada de performance do modelo por `referral source` para validar
robustez em subgrupos. Objetivo: garantir que o modelo generaliza para pacientes
de fontes conhecidas (SVI, SVHC, STMW, SVHD) sem depender do padrao problematico
de "other".

## Atualização do Pipeline de Preprocessing

1. Limpeza de `?` para NaN + conversao numerica das colunas de exames
2. Encoding do target (binaryClass)
3. Encoding das binarias (t/f e M/F)
4. Feature engineering (risk_factors, psych_lithium_group)
5. Drops (variaveis confirmadas + absorvidas + leakage)
6. Transformacao log1p
7. Tratamento de outlier de age (deterministico)
8. Split treino/teste (estratificado)
9. Export do artefato intermediario

OneHotEncoder, RobustScaler e KNNImputer movidos para o Pipeline do notebook 03.


In [24]:
# validação do shape do dataframe
df.shape

(3772, 33)

## Etapa 5 - Drops

Remocao consolidada das variaveis descartadas por decisoes tomadas no EDA e
durante o feature engineering (etapa 4).

### Variaveis removidas

| Variavel | Categoria | Motivo |
|----------|-----------|--------|
| TT4 | Multicolinearidade | Correlacao 0.79 com FTI, poder preditivo inferior (0.29 vs 0.33) |
| I131 treatment | Sem sinal | Poder preditivo nulo, H2 refutada na camada 4 do EDA |
| query on thyroxine | Categoria rara | Baixa representatividade |
| hypopituitary | Categoria rara | Apenas 1 paciente |
| lithium | Volume insuficiente | 18 pacientes, informacao preservada em psych_lithium_group |
| psych | multicolinearidade | informacao preservada em psych_lithium_group |
| TBG | 100% nulos | Sem dados uteis |
| TBG measured | 100% nulos | Sem dados uteis |
| pregnant | Absorvida | Componente de risk_factors, 100% positivo individualmente (leakage) |
| goitre | Absorvida | Componente de risk_factors, 100% positivo individualmente (leakage) |
| tumor | Absorvida | Componente de risk_factors, proporcao identica ao target |
| num_exames_coletados | Leakage | 179 pacientes 100% positivo, artefato de referral_source=other |
| TSH measured | Leakage | 369 pacientes 100% positivo |
| TT4 measured | Leakage | 231 pacientes 97.8% positivo (5.5 pontos acima do baseline) |

### Variaveis mantidas apos deliberacao

- **T3 measured, T4U measured, FTI measured:** nao apresentaram leakage individual,
mantidas para avaliacao no notebook 03
- **on antithyroid medication:** tratamento especifico para hipertireoidismo, sinal
clinico valido
- **on thyroxine:** apesar de flagada como leakage potencial no EDA, sera avaliada
no notebook 03 antes de decisao final

### Validacao esperada

- Shape do dataframe apos drops: (3772, 19) onde 19 reflete o total de colunas
remanescentes apos remocao das 13 variaveis listadas
- Nenhum NaN inesperado gerado pelo drop
- Colunas mantidas conforme lista acima

## Remocao adicional: on thyroxine (leakage confirmado na modelagem)

Esta remocao nao fazia parte das decisoes originais do EDA. Foi incluida apos a
investigacao de leakage conduzida no notebook 03.

Motivo: on thyroxine e tratamento para hipotireoidismo, condicao oposta ao alvo
(hipertireoidismo). Na inspecao dos coeficientes da LogisticRegression, apareceu
como maior peso do modelo (coef [4.68]), indicando que o modelo se apoiava
nessa feature como atalho em vez de aprender o quadro clinico.

A remocao foi validada empiricamente: retirar on thyroxine reduziu o recall de
[0.9935] para [0.9897], queda minima, confirmando que a feature era leakage clinico, porem redundante. Detalhe completo da investigacao no notebook 03.

Decisao: remover on thyroxine na origem para manter consistencia entre os artefatos
consumidos pelos notebooks 03 e 04.

In [25]:
# remoção das variaveis descartadas
df = df.drop(columns=['TT4',
                      'I131 treatment',
                      'query on thyroxine',
                      'hypopituitary',
                      'lithium',
                      'psych',
                      'TBG',
                      'TBG measured',
                      'pregnant',
                      'goitre',
                      'tumor',
                      'num_exames_coletados',
                      'TSH measured',
                      'TT4 measured',
                      'on thyroxine'
                      ])

**Nota sobre psych:** o EDA original previa manter psych como variavel individual.
Apos a criacao da feature psych_lithium_group na etapa 4, decidiu-se dropar psych
para evitar multicolinearidade e redundancia informacional. A informacao clinica
esta preservada na feature engineered.


In [26]:
# validação do shape do dataframe
df.shape

(3772, 18)

## Etapa 6 - Transformacao log1p

Aplicacao de transformacao logaritmica (`log1p`) em variaveis continuas com
assimetria significativa a direita, para aproximar a distribuicao de uma gaussiana
e reduzir o impacto de outliers.

### Efeitos da transformacao:

- Comprime valores altos, mantendo a ordem relativa
- Reduz o peso de outliers na distancia euclidiana (importante para KNNImputer e
scaling posteriores)
- Aproxima a distribuicao de uma gaussiana quando ha cauda longa a direita

### Posicionamento no pipeline

Aplicada **antes do split** por ser transformacao deterministica (cada valor e
transformado individualmente, sem dependencia estatistica da base). Nao introduz
risco de vazamento entre treino e teste.

### Criterio de aplicacao

A decisao de aplicar log1p em cada variavel continua e baseada em analise de
assimetria (skewness):

- Skewness proximo de 0: distribuicao simetrica, transformacao nao recomendada
- Skewness > 1: assimetria significativa, transformacao apropriada
- Skewness > 3: assimetria extrema, transformacao essencial

### Analise inicial

as variaveis continuas antes da transformacao:

| Variavel | Skew original | Aplicar log1p? | Justificativa |
|----------|---------------|----------------|---------------|
| TSH | 13.88 | Sim | Skewness extremo, transformacao essencial |
| T3 | 1.73 | Sim | Skewness significativo, transformacao apropriada |
| T4U | 1.23 | Sim | Skewness significativo, transformacao apropriada |
| FTI | 1.34 | Sim | Skewness significativo, transformacao apropriada |
| age | 1.95 | Nao | Bounded, outlier de qualidade sera tratado separadamente |

**obs**: age nao recebera log1p. Apesar do skewness de 1.95, essa assimetria e inflada
por outlier de qualidade de dado (valor 455 detectado no EDA). Age e variavel
bounded (0-120 anos) sem cauda longa natural. O outlier sera tratado por imputacao
apos o split, momento em que o skewness sera reavaliado.

### Variaveis transformadas

Skewness apos aplicacao de log1p:

| Variavel | Skew original | Skew apos log1p | Avaliacao |
|----------|--------------|-----------------|-----------|
| age | 1.96 | 1.96 (nao transformada) | Sera avaliada apos tratamento de outlier |
| TSH | 13.88 | 1.92 | Melhora massiva. Ainda com assimetria moderada, aceitavel. |
| T3 | 1.73 | -0.29 | Distribuicao praticamente simetrica. |
| T4U | 1.23 | 0.63 | Moderadamente positiva, aceitavel. |
| FTI | 1.34 | -3.05 | **Over-correcao. Assimetria negativa extrema.** |

### Validacao esperada

Skewness reduzido para valores proximos de 0 nas variaveis transformadas.
Distribuicao visualmente mais equilibrada em histogramas.

In [27]:
# verificação do skewness
df[['age','TSH','T3','T4U','FTI']].skew()

,0
age,1.955814
TSH,13.882653
T3,1.730874
T4U,1.232674
FTI,1.345432


In [28]:
# tranformação logaritimica das variaveis TSH, T3, T4U, FTI
df[['TSH','T3', 'T4U','FTI' ]] = np.log1p(df[['TSH','T3', 'T4U','FTI' ]])

In [29]:
# verificação do skewness
df[['age','TSH','T3','T4U','FTI']].skew()

,0
age,1.955814
TSH,1.915162
T3,-0.288412
T4U,0.635327
FTI,-3.048225


In [30]:
# refereção do fti, para investigação devido agravamento da assimetria, no sentido oposto
np.expm1(df['FTI']).describe()

,FTI
count,3387.000000
mean,110.469649
std,33.089698
min,2.000000
25%,93.000000
50%,107.000000
75%,124.000000
max,395.000000


In [31]:
fti_original = np.expm1(df['FTI'])
df[fti_original < 40]['binaryClass'].value_counts()

,count
binaryClass,
0,46
1,6


### Investigacao do FTI

A transformacao log1p do FTI resultou em skewness de -3.05, pior que o valor
original de 1.34. Investigacao dos valores baixos foi conduzida para entender
se sao anomalias de dado ou sinal clinico valido.

**Distribuicao original do FTI:**

| Metrica | Valor |
|---------|-------|
| count | 3387 |
| mean | 110.46 |
| std | 33.08 |
| min | 2 |
| 25% | 93 |
| 50% | 107 |
| 75% | 124 |
| max | 395 |

A maioria dos pacientes esta entre 93-124 (proximo do intervalo clinico normal),
mas o minimo de 2 indica valores muito baixos coexistindo na base.

**Corte utilizado para analise:** FTI < 40 (valor abaixo do limite clinico normal
inferior, capturando casos claramente fora da faixa esperada em hipertireoidismo).

**Distribuicao por classe do target no grupo FTI < 40:**

| Classe | Pacientes | Proporcao |
|--------|-----------|-----------|
| 0 (negativo) | 46 | 88.5% |
| 1 (positivo) | 6 | 11.5% |

**Interpretacao clinica:**

FTI baixo indica producao insuficiente de T4 livre, condicao caracteristica
de hipotireoidismo, nao de hipertireoidismo. Como 88.5% dos pacientes com FTI
baixo sao negativos para hipertireoidismo, o padrao e clinicamente coerente:
esses sao provavelmente pacientes hipotireoidianos incluidos como negativos
no dataset.

Os valores baixos de FTI **nao sao erros de dado**, sao sinal clinico valido
de outra condicao tireoidiana.

### Decisao sobre FTI

A transformacao log1p over-corrigiu ao tratar como outliers valores que sao
clinicamente validos. Manter a transformacao criaria distorcao artificial
na distribuicao (skewness -3.05).

**Decisao:** reverter a transformacao no FTI. Aceitar o skewness original de 1.34
(assimetria moderada, dentro do aceitavel para modelos baseados em arvore).

Reversao aplicada via `np.expm1()`.

### Variaveis transformadas (versao final)

| Variavel | Skew original | Skew final | Transformacao |
|----------|---------------|------------|---------------|
| TSH | 13.88 | 1.92 | log1p aplicado |
| T3 | 1.73 | -0.29 | log1p aplicado |
| T4U | 1.23 | 0.63 | log1p aplicado |
| FTI | 1.34 | 1.34 | Nao transformada (revertida apos investigacao) |
| age | 1.96 | 1.96 | Nao transformada (outlier de qualidade sera tratado apos split) |

### Validacao final

Todas as variaveis continuas ou tem skewness moderado (< 2) ou serao tratadas
em etapas posteriores. Distribuicoes agora mais adequadas para os proximos
passos do pipeline (scaling e KNNImputer).

In [32]:
# reversao da tranformação logaritimica de FTI
df['FTI'] = np.expm1(df['FTI'])

In [33]:
# verificação do skewness
df[['age','TSH','T3','T4U','FTI']].skew()

,0
age,1.955814
TSH,1.915162
T3,-0.288412
T4U,0.635327
FTI,1.345432


## Tratamento de outlier de age

Regra deterministica (limiar fixo de 115 anos), portanto pertence ao bloco
deterministico e roda antes do split. A marcacao do NaN fica aqui; o
preenchimento e feito pelo KNNImputer dentro do Pipeline no notebook 03.


In [34]:
# Tratamento de outlier de age (regra deterministica, antes do split)
# age > 115 e clinicamente inviavel; marca como NaN para o KNNImputer resolver no notebook 03
df.loc[df['age'] > 115, 'age'] = np.nan

# validacao
display(df['age'].sort_values(ascending=False).head(5))


,age
1129,94.0
2673,94.0
3014,93.0
2418,93.0
2760,92.0


## Etapa 7 - Split treino/teste

Divisao do dataset em conjuntos de treinamento e teste. Este split marca a
fronteira entre operacoes deterministicas (aplicadas antes) e operacoes
estatisticas (aplicadas depois, com fit no treino e transform em ambos).

### Configuracao do split

| Parametro | Valor | Justificativa |
|-----------|-------|---------------|
| test_size | 0.25 | "25% no teste, resultando em ~73 pacientes negativos. Ver observacao sobre limitacao amostral abaixo |
| random_state | 42 | Reprodutibilidade. Convencao amplamente adotada. |
| stratify | binaryClass | Preserva a proporcao 92.29% positivo em treino e teste. Essencial para dataset com desbalanceamento severo. |


**Obs:** 73 pacientes negativos no teste. Amostra pequena por causa do desbalanceamento
severo do dataset (291 negativos totais). Metricas na classe minoritaria terao
intervalo de confianca amplo, o que sera considerado na interpretacao dos
resultados no notebook 04.

### Fronteira do pipeline

**Antes do split (ja executado):**
- Limpeza de representacao de nulos
- Encoding do target e binarias
- Feature engineering
- Drops
- Transformacao log1p

Todas essas operacoes sao deterministicas: cada valor e transformado
individualmente ou por regras fixas, sem dependencia estatistica da base.

**Depois do split (proximas etapas):**
- Tratamento de outliers
- OneHotEncoder no referral_source
- Scaling nas continuas
- KNNImputer

Todas essas operacoes exigem fit no treino e transform em ambos os conjuntos,
para evitar vazamento de informacao do teste para o treino.

### Sobre correcao de desbalanceamento

Nao aplicada nesta etapa. Tecnicas como SMOTE, class_weight, undersampling e
oversampling serao avaliadas no notebook 03, durante o treinamento. Todas
serao aplicadas exclusivamente ao conjunto de treino, jamais ao teste.

### Validacao esperada

- X_train e y_train com aproximadamente 2829 linhas
- X_test e y_test com aproximadamente 943 linhas
- Proporcao de positivos preservada em ambos: ~92.29%
- Nenhuma sobreposicao de indices entre treino e teste

In [35]:
# separação de teste e treino
X = df.drop(columns=['binaryClass'])
y = df['binaryClass']

# split em teste e treino
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

In [36]:
# validação do shape do dataframe
display(X_train.shape)
display(X_test.shape)
display(y_train.shape)
display(y_test.shape)

(2829, 17)

(943, 17)

(2829,)

(943,)

In [37]:
# validadção da proporcao de positivos em y_train e y_test
display(y_train.value_counts(normalize=True))
display(y_test.value_counts(normalize=True))

,proportion
binaryClass,
1,0.922941
0,0.077059


,proportion
binaryClass,
1,0.922587
0,0.077413


## Estado final e exportacao do artefato

O notebook 02 encerra no estado pos-deterministico e pre-estatistico. As operacoes
estatisticas (OneHotEncoder, RobustScaler, KNNImputer) foram movidas para o Pipeline
do notebook 03, onde rodam por fold no cross-validation para evitar vazamento.

Estado esperado do artefato exportado:

- X_train / X_test com 17 [~18~] colunas
- referral source ainda como texto (object), sera codificada no Pipeline
- NaN preservados em age e nos labs, serao imputados no Pipeline
- sem escala aplicada


In [38]:
# validacao do estado do artefato antes do export
print('X_train:', X_train.shape, '| X_test:', X_test.shape)
print('referral source dtype:', X_train['referral source'].dtype)   # esperado: object
print('NaN em X_train:', X_train.isna().sum().sum())                 # esperado: > 0
print('NaN em X_test:', X_test.isna().sum().sum())                   # esperado: > 0


X_train: (2829, 17) | X_test: (943, 17)
referral source dtype: object
NaN em X_train: 1569
NaN em X_test: 493


In [39]:
# export do csv dos x_train, X_test, y_train, y_test

X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

In [40]:
# X_train.to_csv('../data/processed/X_train.csv', index=False)
# X_test.to_csv('../data/processed/X_test.csv', index=False)
# y_train.to_csv('../data/processed/y_train.csv', index=False)
# y_test.to_csv('../data/processed/y_test.csv', index=False)